# PF4 Genomic Region Definition using Ensembl REST

This notebook retrieves the genomic coordinates of the PF4 gene using the Ensembl REST API. It extracts gene-level information (e.g., chromosome, start/end positions, strand, and assembly) and defines a genomic region of interest by extending the gene boundaries by ±50 kb.

In [1]:
import requests
import pandas as pd
import json
from pathlib import Path

In [2]:
out_dir = Path("../data/interim/ensembl")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "region.csv"
out_json = out_dir / "region.json"

In [3]:
server = "https://rest.ensembl.org"
species = "homo_sapiens"
gene_symbol = "PF4"

endpoint = f"/lookup/symbol/{species}/{gene_symbol}"

headers = {
    "Content-Type": "application/json"
}

In [4]:
response = requests.get(server + endpoint, headers=headers)

if not response.ok:
    response.raise_for_status()

pf4_data = response.json()

pf4_data

{'db_type': 'core',
 'source': 'ensembl_havana',
 'display_name': 'PF4',
 'description': 'platelet factor 4 [Source:HGNC Symbol;Acc:HGNC:8861]',
 'logic_name': 'ensembl_havana_gene_homo_sapiens',
 'object_type': 'Gene',
 'seq_region_name': '4',
 'version': 5,
 'strand': -1,
 'species': 'homo_sapiens',
 'start': 73980811,
 'assembly_name': 'GRCh38',
 'biotype': 'protein_coding',
 'end': 73982027,
 'id': 'ENSG00000163737',
 'canonical_transcript': 'ENST00000296029.4'}

In [5]:
gene_info = {
    "gene_symbol": gene_symbol,
    "ensembl_gene_id": pf4_data.get("id"),
    "display_name": pf4_data.get("display_name"),
    "species": species,
    "chromosome": pf4_data.get("seq_region_name"),
    "gene_start": pf4_data.get("start"),
    "gene_end": pf4_data.get("end"),
    "strand": pf4_data.get("strand"),
    "assembly_name": pf4_data.get("assembly_name"),
    "biotype": pf4_data.get("biotype"),
    "source": "Ensembl REST lookup/symbol",
}

gene_info

{'gene_symbol': 'PF4',
 'ensembl_gene_id': 'ENSG00000163737',
 'display_name': 'PF4',
 'species': 'homo_sapiens',
 'chromosome': '4',
 'gene_start': 73980811,
 'gene_end': 73982027,
 'strand': -1,
 'assembly_name': 'GRCh38',
 'biotype': 'protein_coding',
 'source': 'Ensembl REST lookup/symbol'}

In [6]:
flanking_bp = 50_000

region_info = gene_info.copy()
region_info["flanking_bp"] = flanking_bp
region_info["region_start"] = max(1, region_info["gene_start"] - flanking_bp)
region_info["region_end"] = region_info["gene_end"] + flanking_bp

region_info

{'gene_symbol': 'PF4',
 'ensembl_gene_id': 'ENSG00000163737',
 'display_name': 'PF4',
 'species': 'homo_sapiens',
 'chromosome': '4',
 'gene_start': 73980811,
 'gene_end': 73982027,
 'strand': -1,
 'assembly_name': 'GRCh38',
 'biotype': 'protein_coding',
 'source': 'Ensembl REST lookup/symbol',
 'flanking_bp': 50000,
 'region_start': 73930811,
 'region_end': 74032027}

In [7]:
region_df = pd.DataFrame([region_info])

region_df

,gene_symbol,ensembl_gene_id,display_name,species,chromosome,gene_start,gene_end,strand,assembly_name,biotype,source,flanking_bp,region_start,region_end
0,PF4,ENSG00000163737,PF4,homo_sapiens,4,73980811,73982027,-1,GRCh38,protein_coding,Ensembl REST lookup/symbol,50000,73930811,74032027


In [8]:
region_df.to_csv(out_csv, index=False)

with open(out_json, "w") as f:
    json.dump(region_info, f, indent=4)